# FAISS Tutorial: Vector Similarity Search & Embeddings

This notebook demonstrates how to perform vector similarity search using **FAISS** (Facebook AI Similarity Search) and **Sentence Transformers** (`all-mpnet-base-v2`).

## 1. Install Packages

In [1]:
#Install Packages
!pip install faiss-cpu
!pip install sentence-transformers

   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/16.3 MB 10.1 MB/s eta 0:00:02
   ----------- ---------------------------- 4.7/16.3 MB 13.6 MB/s eta 0:00:01
   --------------------- ------------------ 8.9/16.3 MB 15.4 MB/s eta 0:00:01
   ----------------------------- ---------- 12.1/16.3 MB 15.1 MB/s eta 0:00:01
   ---------------------------------------  16.3/16.3 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 16.3/16.3 MB 15.5 MB/s  0:00:01



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------- ----------- 524.3/740.6 kB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 740.6/740.6 kB 2.4 MB/s  0:00:00
   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   ---------------------------------------- 846.4/846.4 kB 9.3 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------------------------------- -------- 3.1/4.0 MB 15.3 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 14.2 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   --------- ------------------------------ 2.9/12.3 MB 14.0 MB/s eta 0:00:01
   -------------------- ------------------- 6.3/12.3 MB 15.4 MB/s eta 0:00:01
   -------------------------------- ------- 10.0/12.3 MB 16.3 MB/s eta 0:00:01
   ----------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 3.6.0 requires cryptography<47,>=43.0.0, but you have cryptography 50.0.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import Necessary Libraries

In [2]:
# import necessary libraries
import pandas as pd
pd.set_option('display.max_colwidth', 100)

## 3. Load & Inspect Dataset

In [3]:
df = pd.read_csv("sample_text.csv")
df.shape

(8, 2)

In [4]:
df 

,text,category
0,Meditation and yoga can improve mental health,Health
1,"Fruits, whole grains and vegetables helps control blood pressure",Health
2,These are the latest fashion trends for this week,Fashion
3,Vibrant color jeans for male are becoming a trend,Fashion
4,The concert starts at 7 PM tonight,Event
5,Navaratri dandiya program at Expo center in Mumbai this october,Event
6,Exciting vacation destinations for your next trip,Travel
7,Maldives and Srilanka are gaining popularity in terms of low budget vacation places,Travel


## 4. Vector Embeddings & Index Building

### Step 1 : Create source embeddings for the text column

In [5]:
from sentence_transformers import SentenceTransformer

In [8]:
encoder = SentenceTransformer("all-mpnet-base-v2")
vectors = encoder.encode(df["text"].tolist())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
vectors.shape

(8, 768)

In [10]:
vectors

array([[-0.00247393,  0.03626724, -0.05290457, ..., -0.09152355,
        -0.03970002, -0.04330488],
       [-0.03357266,  0.00980519, -0.03250128, ..., -0.05165466,
         0.02245887, -0.03156182],
       [-0.01865324, -0.04051313, -0.01235388, ...,  0.00610587,
        -0.07179645,  0.02773851],
       ...,
       [-0.00066459,  0.04252128, -0.05645507, ...,  0.01315472,
        -0.03183567, -0.04357663],
       [-0.03317154,  0.03252457, -0.02484838, ...,  0.01174421,
         0.05747123,  0.00571022],
       [-0.00166394,  0.00413825, -0.04597083, ...,  0.02008527,
         0.05656242, -0.00161596]], dtype=float32)

In [13]:
dim = vectors.shape[1]
dim

768

### Step 2 : Build a FAISS Index for vectors

In [14]:
import faiss

index = faiss.IndexFlatL2(dim)

### Step 3 : Normalize the source vectors (as we are using L2 distance to measure similarity) and add to the index

In [15]:
index.add(vectors)

In [16]:
index

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001ADDD8A1570> >

## 5. Querying & Similarity Search

### Step 4 : Encode search text using same encorder and normalize the output vector

In [17]:
search_query = "I want to buy a polo t-shirt"
# search_query = "looking for places to visit during the holidays"
# search_query = "An apple a day keeps the doctor away"
vec = encoder.encode(search_query)
vec.shape

(768,)

In [18]:
import numpy as np
svec = np.array(vec).reshape(1,-1)
svec.shape

(1, 768)

### Step 5: Search for similar vector in the FAISS index created

In [20]:
distances, I = index.search(svec, k=2)
distances

array([[1.3844836, 1.4039092]], dtype=float32)

In [21]:
I

array([[3, 2]], dtype=int64)

### Step 6: Retrieve and Inspect the Most Similar Documents

In [23]:
df.loc[[3,2]]

,text,category
3,Vibrant color jeans for male are becoming a trend,Fashion
2,These are the latest fashion trends for this week,Fashion


In [24]:
search_query

'I want to buy a polo t-shirt'

In [25]:
df

,text,category
0,Meditation and yoga can improve mental health,Health
1,"Fruits, whole grains and vegetables helps control blood pressure",Health
2,These are the latest fashion trends for this week,Fashion
3,Vibrant color jeans for male are becoming a trend,Fashion
4,The concert starts at 7 PM tonight,Event
5,Navaratri dandiya program at Expo center in Mumbai this october,Event
6,Exciting vacation destinations for your next trip,Travel
7,Maldives and Srilanka are gaining popularity in terms of low budget vacation places,Travel


You can see that the two results from the dataframe are similar to a search_query